# Swarm Demo: Multi-Agent Log Analysis with Memory Pointer Pattern

## The Problem

When an AI agent fetches large data — like application logs — the tool output enters the LLM's context window. With 145KB+ of log data, this overflows the context limit and breaks the workflow.

Without a solution:
```
Agent → fetch_logs() → 145KB JSON → enters LLM context → ❌ context overflow
```

## The Solution

Store large data in **`invocation_state`** — shared memory outside the context window. Tools write data there and return only a small pointer string. The LLM never sees the raw data.

```
Collector → fetch_logs() → stores 145KB in invocation_state → returns "logs-payment-service"
Analyzer  → analyze_errors("logs-payment-service") → reads from invocation_state → stores results
Reporter  → generate_report() → reads results from invocation_state → produces final report
```

**Key API:** `tool_context.invocation_state` — the official Strands API for sharing state across agents.

See: [Shared State Across Multi-Agent Patterns](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/multi-agent/multi-agent-patterns/)

In [ ]:
# Setup: load tools, agents, and the swarm
#
# What gets imported:
#   Tools (5):
#     - fetch_application_logs   → generates 600 log events, stores them in invocation_state
#     - analyze_error_patterns   → reads logs pointer, counts errors by service
#     - detect_latency_anomalies → reads logs pointer, computes p95 latency
#     - generate_incident_report → reads both analyses, produces final report
#     - get_error_details        → drill into errors for a specific service
#
#   Agents (3):
#     - collector  → runs fetch_application_logs, then hands off to analyzer
#     - analyzer   → runs both analysis tools, then hands off to reporter
#     - reporter   → runs generate_incident_report, delivers result to user
#
#   Swarm (1):
#     - orchestrates the collector → analyzer → reporter pipeline automatically

import os
os.environ['OTEL_SDK_DISABLED'] = 'true'

from dotenv import load_dotenv
load_dotenv()

from swarm_demo_tools import (
    fetch_application_logs, analyze_error_patterns,
    detect_latency_anomalies, generate_incident_report, get_error_details,
    collector, analyzer, reporter, swarm, MODEL
)

print("✅ Swarm ready: collector → analyzer → reporter")

## Step 1 — Run the Swarm

One request triggers the full pipeline. While it runs, watch for:

- **Tool calls printed**: each agent calls only its own tools — the collector never calls `generate_incident_report`, the reporter never calls `fetch_logs`
- **Handoffs**: `node_history` in the result shows which agents ran and in what order
- **What the LLM sees**: pointer strings like `"logs-payment-service"`, not 145KB of raw JSON

The swarm manages all coordination — no manual handoff code needed.

## Run the Swarm

One request triggers the full pipeline. The 145KB+ of logs flow through `invocation_state` — never entering any LLM context.

---

## Step 2 — Follow-up Investigation

The swarm finished — but `invocation_state` still holds everything: the 145KB of logs, the error analysis, and the latency analysis.

A single investigator agent can now ask follow-up questions **without re-fetching any data**. It reads directly from the same stored state.

**Why this matters:** In a traditional approach, each new question would trigger a full re-fetch of 145KB of logs. With `invocation_state`, the data persists for the lifetime of the workflow — any agent can access it at any time, as many times as needed.

The three turns below simulate a real investigation conversation: identify the worst service, inspect its logs, then diagnose the error type.

In [ ]:
result = swarm("Fetch 6 hours of logs for payment-service, analyze errors and latency, then generate an incident report.")

print(f"\nStatus: {result.status}")
print(f"Agents: {' → '.join(n.node_id for n in result.node_history)}")
print(f"Time: {result.execution_time}ms")

---

## 💬 Follow-up: Investigate the Problem

The swarm completed and the data is still in `invocation_state`. Now we can use a single investigator agent to drill into specific services — the 145KB of logs are still available without re-fetching.

In [ ]:
from strands import Agent

investigator = Agent(
    name="investigator",
    system_prompt="You investigate incidents. The logs pointer is 'logs-payment-service'. Use get_error_details to drill into specific services.",
    tools=[get_error_details, analyze_error_patterns],
    model=MODEL,
)

print("💬 Follow-up investigation with stored data\n")
print("="*60)

# Turn 1
print("\n👤 User: Which service had the most errors?\n")
investigator("Based on the error analysis in shared state, which service had the most errors? The logs are at 'logs-payment-service'")

# Turn 2
print("\n" + "="*60)
print("\n👤 User: Show me the actual error logs for that service\n")
investigator("Show me 3 detailed error logs for the service with the most errors")

# Turn 3
print("\n" + "="*60)
print("\n👤 User: What status codes are those errors?\n")
investigator("What HTTP status codes are those errors returning? Are they 500s or 400s?")

print("\n" + "="*60)
print("\n📦 Data persisted in invocation_state throughout all queries")
print("   (never re-fetched, never entered LLM context)")

---

## Key Takeaways

1. **Swarm handles coordination** — collector → analyzer → reporter with autonomous handoffs
2. **invocation_state for multi-agent data** — `tool_context.invocation_state` is the official Strands API for sharing data across agents
3. **Large data stays out of context** — 145KB+ of logs in invocation_state, only pointers in LLM context
4. **Data persists after swarm completes** — follow-up investigation reuses the same stored data
5. **Same ToolContext API** — single-agent uses `agent.state`, multi-agent uses `invocation_state`, both via `ToolContext`

## References

- [Strands Swarm](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/multi-agent/swarm/) — Multi-agent orchestration
- [Shared State Across Multi-Agent Patterns](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/multi-agent/multi-agent-patterns/) — invocation_state for data sharing
- [Strands ToolContext](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/tools/creating-custom-tools/) — Accessing agent.state and invocation_state
- [Solving Context Window Overflow](https://arxiv.org/html/2511.22729v1) — IBM Research
- [Towards Effective GenAI Multi-Agent Collaboration](https://arxiv.org/pdf/2412.05449) — Amazon, payload referencing